In [3]:
!pip install -U langchain-text-splitters
!pip install -U langchain-community langchain-text-splitters langchain
!pip install pypdf
!pip install -U langchain-huggingface
!pip install -qU langchain-chroma
!pip install -qU langchain
!pip install -qU langchain-groq
!pip install sentence-transformers
!pip install -q gradio

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.5.0-py3-none-any.whl (596 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into ac

In [4]:
from langchain_groq import ChatGroq
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
import shutil
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
import re
import gc

In [48]:
from google.colab import drive
from google.colab import userdata

# --- MONTAGE DRIVE ET CHEMINS ---
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Smart-HR-Assistant"
NEW_DATA_PATH = os.path.join(PROJECT_PATH, "nouveau_data")
ACTUEL_DATA_PATH = os.path.join(PROJECT_PATH, "actuel_data")
CHROMA_PATH = os.path.join(PROJECT_PATH, "chroma_db")

# Création des dossiers si nécessaire
for path in [NEW_DATA_PATH, ACTUEL_DATA_PATH, CHROMA_PATH]:
    os.makedirs(path, exist_ok=True)

# --- CONFIGURATION API ---
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def charger_nouveau_data(chemin_repertoire="./nouveau_data"):
    # On vérifie si le dossier existe, sinon on le crée
    if not os.path.exists(chemin_repertoire):
        os.makedirs(chemin_repertoire)
        print(f"Dossier {chemin_repertoire} créé. Ajoutez vos PDF dedans !")
        return []

    # Le DirectoryLoader s'occupe de scanner récursivement
    loader = DirectoryLoader(
        chemin_repertoire,
        glob="**/*.pdf",  # Cherche les PDF même dans les sous-dossiers
        loader_cls=PyPDFLoader
    )

    docs = loader.load()
    print(f"✅ Analyse terminée : {len(docs)} pages trouvées dans {chemin_repertoire}")
    return docs

In [40]:
# Formuler la question
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", (
    "Ton unique job est de reformuler la question de l'utilisateur pour qu'elle soit "
    "compréhensible toute seule, en utilisant l'historique. "
    "REGLÈS : "
    "1. NE RÉPONDS PAS à la question. "
    "2. NE DONNE AUCUNE INFORMATION sur l'entreprise. "
    "3. Renvoie UNIQUEMENT la question reformulée. "
    "Exemple : Si l'historique parle de Mutuelle et que l'utilisateur dit 'C'est combien ?', "
    "tu réponds juste : 'Quel est le prix de la mutuelle ?'"
    )),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()

# Le rédacteur
instruction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Tu es un expert en analyse de documents RH. Ton rôle est de fournir des résumés clairs, "
        "précis et structurés à partir du contexte fourni ci-dessous.\n\n"
        "CONTEXTE :\n{context}\n\n"
        "DIRECTIVES :\n"
        "1. Agis comme un analyste senior : synthétise les points clés, les noms, les dates et les montants importants.\n"
        "2. Si l'information demandée n'est pas présente dans le contexte, dis simplement que tu ne sais pas.\n"
        "3. Réponds de manière professionnelle en utilisant des listes à puces."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

In [41]:
def get_or_update_db():
    os.makedirs(os.path.dirname(CHROMA_PATH), exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # 1. Charger la base existante (ou en créer une vide)
    vector_db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

    # 2. Vérifier s'il y a du nouveau
    new_files = [f for f in os.listdir(NEW_DATA_PATH) if f.endswith('.pdf')]

    if new_files:
        loader = PyPDFDirectoryLoader(NEW_DATA_PATH)
        new_docs = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        new_chunks = text_splitter.split_documents(new_docs)

        # Ajout à Chroma
        vector_db.add_documents(new_chunks)

        # Déplacement vers l'archive
        for file_name in new_files:
            shutil.move(os.path.join(NEW_DATA_PATH, file_name),
                        os.path.join(ACTUEL_DATA_PATH, file_name))
            print(f"Archivé : {file_name}")
    else:
        print("Aucun nouveau fichier. Utilisation de la base existante.")

    return vector_db

In [42]:
def respond(message, chat_history):
    global llm, contextualize_q_chain # On ajoute la chaîne de reformulation

    # 1. Mise à jour de la base (ou récupération)
    vector_db = get_or_update_db()

    # --- NOUVEAU : TRANSFORME L'HISTORIQUE POUR LE LLM ---
    formatted_history = []
    for user_msg, ai_msg in chat_history:
        formatted_history.append(HumanMessage(content=user_msg))
        formatted_history.append(AIMessage(content=ai_msg))

    # --- ÉTAPE 2 : REFORMULATION (Le Secrétaire travaille ici) ---
    # Si c'est la 1ère question, on garde le message tel quel.
    # Sinon, on demande au LLM de clarifier la question par rapport à l'historique.
    if chat_history:
        standalone_question = contextualize_q_chain.invoke({
            "chat_history": formatted_history,
            "input": message
        })
        print(f"DEBUG - Question reformulée : {standalone_question} okk")
    else:
        standalone_question = message

    # --- ÉTAPE 3 : RECHERCHE (On cherche avec la question reformulée) ---
    docs = vector_db.similarity_search(standalone_question, k=3)
    context = "\n\n".join([d.page_content for d in docs])

    # --- ÉTAPE 4 : RÉPONSE FINALE (Le Rédacteur) ---
    chain = instruction_prompt | llm
    result = chain.invoke({
        "context": context,
        "question": message, # On garde la question originale pour l'affichage
        "chat_history": formatted_history
    })

    # 5. Ajouter l'échange à l'historique
    chat_history.append((message, result.content))
    return "", chat_history

In [65]:
import shutil

def upload_files(files):
    # On s'assure que le dossier existe
    if not os.path.exists(NEW_DATA_PATH):
        os.makedirs(NEW_DATA_PATH)

    file_paths = []
    for file in files:
        # file.name contient le chemin temporaire du fichier téléchargé
        nom_fichier = os.path.basename(file.name)
        destination = os.path.join(NEW_DATA_PATH, nom_fichier)

        # On copie le fichier du dossier temporaire vers Google Drive
        shutil.copy(file.name, destination)
        file_paths.append(destination)
    return f"SUCCÈS : {len(file_paths)} fichiers ajoutés. Cliquez maintenant sur Synchroniser."

In [66]:
with gr.Blocks() as demo:
    gr.Markdown("# 📁 Assistant Smart-HR - Espace Administration")

    with gr.Row():
        # Partie Upload pour le RH
        with gr.Column(scale=1):
            file_output = gr.File(file_count="multiple", label="Téléverser des PDF RH")
            upload_button = gr.Button("1. Stocker les fichiers")
            status_label = gr.Textbox(label="Statut du stockage")

            update_db_button = gr.Button("2. Synchroniser la base (IA)", variant="primary")
            db_status = gr.Textbox(label="Statut de l'IA")

        # Partie Chat pour l'utilisateur
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Discussion avec vos documents")
            msg = gr.Textbox(label="Posez votre question RH")
            clear = gr.Button("Effacer l'historique")

    # --- LOGIQUE DES BOUTONS ---

    # Bouton 1 : On stocke les fichiers dans le Drive
    upload_button.click(upload_files, inputs=file_output, outputs=status_label, show_progress="hidden", queue=False)

    # Bouton 2 : On lance la fonction maj_base_de_donnees() qu'on a fait juste avant
    update_db_button.click(get_or_update_db, outputs=db_status)

    # Le reste de ta logique de chat (respond...)
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch(debug=True)

/tmp/ipython-input-11385/2051670086.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")
/tmp/ipython-input-11385/2051670086.py:16: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2f1f55308434960544.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Archivé : BM.pdf
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2f1f55308434960544.gradio.live
